# Pandas Foundations — Cleaning a Real Messy Dataset

This notebook explores Pandas fundamentals through a deliberately messy customer sales dataset. The workflow covers dataset inspection, missing-data diagnosis, labeled selection, Boolean filtering, category cleaning, group-based analysis, and vectorized operations.

The goal is to understand not only how to clean data, but also why each cleaning decision is made.

## Introduction

Real-world datasets rarely arrive in a perfectly clean form. They may contain missing values, inconsistent categorical representations, and other structural problems that must be understood before analysis.

This notebook follows a simple data-analysis workflow:

1. Create and inspect the raw data.
2. Diagnose data-quality problems.
3. Choose and justify appropriate cleaning strategies.
4. Clean the dataset.
5. Filter and analyze the cleaned data.
6. Measure the performance of different Pandas approaches.

The notebook is designed to be reproducible and will be tested by restarting the kernel and running all cells from a fresh state.

In [1]:
import numpy as np
import pandas as pd

## 1. Building a Deliberately Messy Dataset

Before working with a real dataset, we will construct a small sales dataset with known data-quality problems.

The dataset contains transaction dates, product categories, transaction amounts, and customer ages. Missing values and inconsistent category casing are deliberately introduced so that we can diagnose and clean them in later sections.

At this stage, no cleaning will be performed because we first need to understand the problems present in the raw data.

In [2]:
sales_data = {
    "date": [
        "2026-08-01",
        "2026-08-02",
        "2026-08-03",
        "2026-08-04",
        "2026-08-05",
        "2026-08-06",
        "2026-08-07",
        "2026-08-08",
        "2026-08-09",
        "2026-08-10",
    ],
    "category": [
        "Electronics",
        "Clothing",
        "electronics",
        "Furniture",
        None,
        "Clothing",
        "Electronics",
        "Furniture",
        "electronics",
        "Clothing",
    ],
    "amount": [
        1200,
        450,
        None,
        800,
        300,
        None,
        1500,
        650,
        900,
        550,
    ],
    "customer_age": [
        25,
        31,
        28,
        None,
        42,
        35,
        None,
        29,
        24,
        38,
    ],
}

df = pd.DataFrame(sales_data)

df

,date,category,amount,customer_age
0,2026-08-01,Electronics,1200.0,25.0
1,2026-08-02,Clothing,450.0,31.0
2,2026-08-03,electronics,NaN,28.0
3,2026-08-04,Furniture,800.0,NaN
4,2026-08-05,NaN,300.0,42.0
5,2026-08-06,Clothing,NaN,35.0
6,2026-08-07,Electronics,1500.0,NaN
7,2026-08-08,Furniture,650.0,29.0
8,2026-08-09,electronics,900.0,24.0
9,2026-08-10,Clothing,550.0,38.0


### Initial Observation

The dataset contains several deliberate data-quality issues. Some values are missing in the `category`, `amount`, and `customer_age` columns. The `category` column also contains inconsistent capitalization, where `"Electronics"` and `"electronics"` represent the same category but are currently treated as different values.

These issues will be diagnosed quantitatively before any cleaning operation is applied.

### Initial Data Inspection

The `head()` method provides a quick view of the first rows and helps us confirm the structure and values of the raw dataset.

In [3]:
df.head()

,date,category,amount,customer_age
0,2026-08-01,Electronics,1200.0,25.0
1,2026-08-02,Clothing,450.0,31.0
2,2026-08-03,electronics,NaN,28.0
3,2026-08-04,Furniture,800.0,NaN
4,2026-08-05,NaN,300.0,42.0


### DataFrame Structure and Data Types

The `info()` method provides the number of entries, column names, data types, and non-null counts. The non-null counts are particularly useful for identifying columns containing missing values.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          10 non-null     str    
 1   category      9 non-null      str    
 2   amount        8 non-null      float64
 3   customer_age  8 non-null      float64
dtypes: float64(2), str(2)
memory usage: 452.0 bytes


### Dataset Dimensions

The `shape` property tells us how many observations and columns are present in the dataset.

In [5]:
print("Dataset shape:", df.shape)

Dataset shape: (10, 4)


### Numerical Summary

The `describe()` method provides summary statistics for numerical columns. This helps us understand the distribution of the available values before making any decisions about missing data or further analysis.

In [6]:
df.describe()

,amount,customer_age
count,8.000000,8.000000
mean,793.750000,31.500000
std,399.497452,6.347103
min,300.000000,24.000000
25%,525.000000,27.250000
50%,725.000000,30.000000
75%,975.000000,35.750000
max,1500.000000,42.000000


### Missing-Value Diagnosis

The `isna().sum()` operation counts missing values in each column. This gives us a quantitative view of the missing-data problem rather than relying only on visual inspection.

In [7]:
missing_values = df.isna().sum()

print("Missing values per column:")
print(missing_values)

Missing values per column:
date            0
category        1
amount          2
customer_age    2
dtype: int64


In [8]:
df["category"]

0    Electronics
1       Clothing
2    electronics
3      Furniture
4            NaN
5       Clothing
6    Electronics
7      Furniture
8    electronics
9       Clothing
Name: category, dtype: str

### Category Observation

The `category` column contains both `"Electronics"` and `"electronics"`. Although Pandas treats these as different string values, they represent the same category conceptually. This inconsistency will be handled in a later cleaning step.

### Diagnosis of the Raw Dataset

The dataset contains 10 rows and 4 columns. The `category` column contains one missing value, while `amount` and `customer_age` each contain two missing values. The `date` column has no missing values.

There is also a category inconsistency: `"Electronics"` and `"electronics"` appear as separate values even though they represent the same category.

The numerical summary shows that the available values in the `amount` and `customer_age` columns can be inspected before deciding how their missing values should be handled.

At this stage, no values have been modified. The next step is to decide an appropriate missing-data strategy for each affected column.

## 3. Missing-Data Strategy

The previous diagnostic step identified missing values in the `category`, `amount`, and `customer_age` columns.

Instead of applying one rule to the entire dataset, we will decide how to handle missing values separately for each column. The choice will depend on the meaning of the column and whether a reasonable replacement can be justified.

No cleaning will be performed until the strategy for each column has been documented.

### Strategy for `category`

The `category` column contains one missing value. A missing category cannot be reliably inferred from the available information because we do not know which product category the transaction belongs to.

For this dataset, I will use `dropna()` for the row with the missing category rather than assigning an arbitrary category. This prevents us from introducing unsupported information into the categorical data.

The tradeoff is that one transaction will be removed from the dataset.

### Strategy for `amount`

The `amount` column contains two missing values. Because the column is numerical, we can replace the missing values with a representative statistic.

I will use the median amount rather than the mean because transaction amounts can be affected by unusually large purchases, and the median is less sensitive to extreme values.

This preserves the transactions while making an explicit assumption about how to represent the missing amounts.

### Strategy for `customer_age`

The `customer_age` column contains two missing values. Since age is numerical and the missing values do not prevent us from retaining the transactions, I will fill them using the median customer age.

Using the median avoids removing otherwise useful sales records while reducing the influence of unusually young or old customers.

In [9]:
cleaned_df = df.copy()

In [10]:
cleaned_df = cleaned_df.dropna(subset=["category"])

In [11]:
amount_median = cleaned_df["amount"].median()

cleaned_df["amount"] = cleaned_df["amount"].fillna(amount_median)

In [12]:
print("Amount median used for filling:", amount_median)

Amount median used for filling: 800.0


In [13]:
age_median = cleaned_df["customer_age"].median()

cleaned_df["customer_age"] = cleaned_df["customer_age"].fillna(age_median)

print("Customer age median used for filling:", age_median)

Customer age median used for filling: 29.0


### Verifying the Missing-Data Strategy

After applying the selected strategies, we verify the remaining missing values. The `category` row containing the missing value should have been removed, while missing `amount` and `customer_age` values should have been replaced with their respective medians.

In [14]:
print("Missing values after cleaning:")
print(cleaned_df.isna().sum())

Missing values after cleaning:
date            0
category        0
amount          0
customer_age    0
dtype: int64


In [15]:
print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)

Original shape: (10, 4)
Cleaned shape: (9, 4)


### Interpretation

The missing-data strategy was applied separately according to the meaning of each column. The row with the missing `category` was removed because the correct category could not be reliably inferred. Missing `amount` and `customer_age` values were filled using their respective medians so that otherwise useful transactions were preserved.

The cleaned dataset contains no missing values, but it is one row smaller than the original dataset because of the missing category. This demonstrates that missing-data handling involves tradeoffs rather than a single universally correct rule.

## 4. `.loc` vs `.iloc`

Pandas provides two important indexing methods: `.loc` for label-based selection and `.iloc` for position-based selection.

These can initially appear to behave the same when the DataFrame index is in its default order. However, after sorting or filtering, the difference becomes important.

This section demonstrates that difference experimentally.

### Current DataFrame Index

Before comparing `.loc` and `.iloc`, we inspect the current index of the cleaned DataFrame.

In [16]:
cleaned_df

,date,category,amount,customer_age
0,2026-08-01,Electronics,1200.0,25.0
1,2026-08-02,Clothing,450.0,31.0
2,2026-08-03,electronics,800.0,28.0
3,2026-08-04,Furniture,800.0,29.0
5,2026-08-06,Clothing,800.0,35.0
6,2026-08-07,Electronics,1500.0,29.0
7,2026-08-08,Furniture,650.0,29.0
8,2026-08-09,electronics,900.0,24.0
9,2026-08-10,Clothing,550.0,38.0


### Label-Based Selection with `.loc`

`.loc` selects rows using their index labels. Therefore, when we provide `0`, Pandas searches for the row whose index label is `0`.

In [17]:
cleaned_df.loc[0]

date             2026-08-01
category        Electronics
amount               1200.0
customer_age           25.0
Name: 0, dtype: object

### Position-Based Selection with `.iloc`

`.iloc` selects rows according to their integer position. Therefore, `.iloc[0]` means the first row currently displayed in the DataFrame, regardless of its index label.

In [18]:
cleaned_df.iloc[0]

date             2026-08-01
category        Electronics
amount               1200.0
customer_age           25.0
Name: 0, dtype: object

In [19]:
sorted_df = cleaned_df.sort_values("amount", ascending=False)

In [20]:
sorted_df

,date,category,amount,customer_age
6,2026-08-07,Electronics,1500.0,29.0
0,2026-08-01,Electronics,1200.0,25.0
8,2026-08-09,electronics,900.0,24.0
2,2026-08-03,electronics,800.0,28.0
3,2026-08-04,Furniture,800.0,29.0
5,2026-08-06,Clothing,800.0,35.0
7,2026-08-08,Furniture,650.0,29.0
9,2026-08-10,Clothing,550.0,38.0
1,2026-08-02,Clothing,450.0,31.0


### `.loc` After Sorting

After sorting, the index labels remain associated with their original rows. Therefore, `.loc[0]` still selects the row whose label is `0`, even though that row may no longer be the first row.

In [21]:
sorted_df.loc[0]

date             2026-08-01
category        Electronics
amount               1200.0
customer_age           25.0
Name: 0, dtype: object

### `.iloc` After Sorting

`.iloc[0]` selects the first row by its current position. Because the DataFrame has been sorted, this may now be a different row from the one returned by `.loc[0]`.

In [22]:
sorted_df.iloc[0]

date             2026-08-07
category        Electronics
amount               1500.0
customer_age           29.0
Name: 6, dtype: object

In [23]:
print("Row selected by .loc[0]:")
print(sorted_df.loc[0, "amount"])

print("\nRow selected by .iloc[0]:")
print(sorted_df.iloc[0]["amount"])

Row selected by .loc[0]:
1200.0

Row selected by .iloc[0]:
1500.0


In [24]:
loc_row = sorted_df.loc[0]
iloc_row = sorted_df.iloc[0]

print("Index label selected by .loc:", loc_row.name)
print("Index label of row selected by .iloc:", iloc_row.name)

print("\nAmount selected by .loc:", loc_row["amount"])
print("Amount selected by .iloc:", iloc_row["amount"])

Index label selected by .loc: 0
Index label of row selected by .iloc: 6

Amount selected by .loc: 1200.0
Amount selected by .iloc: 1500.0


### Interpretation

The experiment demonstrates that `.loc` and `.iloc` use different selection rules. `.loc[0]` selects the row whose index label is `0`, while `.iloc[0]` selects the first row by its current position.

Before sorting, these operations can appear equivalent because the index labels and row positions happen to align. After sorting, the row positions change while the original labels remain attached to their rows, causing `.loc[0]` and `.iloc[0]` to return different rows.

Therefore:

- `.loc` → label-based indexing
- `.iloc` → position-based indexing

In [25]:
reset_df = sorted_df.reset_index(drop=True)

In [26]:
reset_df

,date,category,amount,customer_age
0,2026-08-07,Electronics,1500.0,29.0
1,2026-08-01,Electronics,1200.0,25.0
2,2026-08-09,electronics,900.0,24.0
3,2026-08-03,electronics,800.0,28.0
4,2026-08-04,Furniture,800.0,29.0
5,2026-08-06,Clothing,800.0,35.0
6,2026-08-08,Furniture,650.0,29.0
7,2026-08-10,Clothing,550.0,38.0
8,2026-08-02,Clothing,450.0,31.0


### Resetting the Index

`reset_index(drop=True)` creates a new sequential index after sorting. This can be useful when the original index labels are no longer meaningful for the cleaned or transformed dataset.

## 5. Boolean Filtering

Boolean filtering allows us to select rows that satisfy a condition. This is closely related to the Boolean masking technique used with NumPy arrays.

In Pandas, conditions are evaluated element-by-element across a Series. Multiple conditions are combined using `&` for AND and `|` for OR.

### Filtering by Transaction Amount

We will select transactions where the amount is greater than 500. This creates a Boolean condition for each row and uses it to filter the DataFrame.

In [27]:
high_value_sales = cleaned_df[cleaned_df["amount"] > 500]

high_value_sales

,date,category,amount,customer_age
0,2026-08-01,Electronics,1200.0,25.0
2,2026-08-03,electronics,800.0,28.0
3,2026-08-04,Furniture,800.0,29.0
5,2026-08-06,Clothing,800.0,35.0
6,2026-08-07,Electronics,1500.0,29.0
7,2026-08-08,Furniture,650.0,29.0
8,2026-08-09,electronics,900.0,24.0
9,2026-08-10,Clothing,550.0,38.0


In [28]:

amount_mask = cleaned_df["amount"] > 500

print(amount_mask)

0     True
1    False
2     True
3     True
5     True
6     True
7     True
8     True
9     True
Name: amount, dtype: bool


### Combining Conditions with AND

We can combine multiple row-level conditions using `&`. The resulting filter keeps only rows where both conditions are true.

In [29]:
filtered_sales = cleaned_df[
    (cleaned_df["amount"] > 500)
    & (cleaned_df["customer_age"] > 30)
]

filtered_sales

,date,category,amount,customer_age
5,2026-08-06,Clothing,800.0,35.0
9,2026-08-10,Clothing,550.0,38.0


### Combining Conditions with OR

The `|` operator represents OR for element-wise Pandas conditions. A row is retained when at least one of the conditions evaluates to `True`.

In [30]:
selected_sales = cleaned_df[
    (cleaned_df["amount"] > 1000)
    | (cleaned_df["category"] == "Clothing")
]

selected_sales

,date,category,amount,customer_age
0,2026-08-01,Electronics,1200.0,25.0
1,2026-08-02,Clothing,450.0,31.0
5,2026-08-06,Clothing,800.0,35.0
6,2026-08-07,Electronics,1500.0,29.0
9,2026-08-10,Clothing,550.0,38.0


In [31]:
# cleaned_df[
#     (cleaned_df["amount"] > 500)
#     and (cleaned_df["customer_age"] > 30)
# ]

### Why Does `and` Fail?

The Python `and` operator expects each side to resolve to one overall Boolean value, such as `True` or `False`.

However, a Pandas condition such as `cleaned_df["amount"] > 500` produces a Boolean Series containing one Boolean value for each row.

Pandas therefore cannot interpret the entire Series as one single truth value. For element-wise logical operations between Series, Pandas uses `&` for AND and `|` for OR.

In [32]:
amount_condition = cleaned_df["amount"] > 500
age_condition = cleaned_df["customer_age"] > 30

print("Amount condition:")
print(amount_condition)

print("\nAge condition:")
print(age_condition)

Amount condition:
0     True
1    False
2     True
3     True
5     True
6     True
7     True
8     True
9     True
Name: amount, dtype: bool

Age condition:
0    False
1     True
2    False
3    False
5     True
6    False
7    False
8    False
9     True
Name: customer_age, dtype: bool


In [33]:
combined_condition = amount_condition & age_condition

print(combined_condition)

0    False
1    False
2    False
3    False
5     True
6    False
7    False
8    False
9     True
dtype: bool


In [34]:
cleaned_df[combined_condition]

,date,category,amount,customer_age
5,2026-08-06,Clothing,800.0,35.0
9,2026-08-10,Clothing,550.0,38.0


### Interpretation

Boolean filtering in Pandas works by creating a Boolean value for each row and using that Boolean Series as a mask.

A single condition such as `amount > 500` selects rows based on one criterion. Multiple conditions can be combined element-wise using `&` for AND and `|` for OR.

The experiment with Python's `and` operator produced a `ValueError` because `and` expects a single Boolean truth value, while Pandas conditions produce a Boolean Series containing one value per row.

Therefore, the correct Pandas pattern is:

- `&` → element-wise AND
- `|` → element-wise OR
- Parentheses → group each individual condition

## 6. Discovering and Cleaning Inconsistent Categories

Categorical data can contain multiple representations of the same concept. In this dataset, `"Electronics"` and `"electronics"` represent the same category but differ in capitalization.

We will first inspect the unique category frequencies using `value_counts()`. After identifying the inconsistency, we will standardize the category values using a vectorized string operation and verify the result.

### Before Cleaning

The `value_counts()` method shows how frequently each distinct category value appears. This allows us to identify inconsistent representations before modifying the data.

In [35]:
cleaned_df["category"].value_counts()

category
Clothing       3
Electronics    2
electronics    2
Furniture      2
Name: count, dtype: int64

In [36]:
print("Unique categories before cleaning:")
print(cleaned_df["category"].unique())

Unique categories before cleaning:
<StringArray>
['Electronics', 'Clothing', 'electronics', 'Furniture']
Length: 4, dtype: str


### Cleaning Strategy

The category values differ only in capitalization, so they represent the same underlying categories. To make the values consistent, we will convert every category to lowercase.

This transformation is applied to the entire column using Pandas' vectorized string operations rather than manually changing individual rows.

In [37]:
cleaned_df["category"] = cleaned_df["category"].str.lower()

In [38]:
print("Unique categories after cleaning:")
print(cleaned_df["category"].unique())

Unique categories after cleaning:
<StringArray>
['electronics', 'clothing', 'furniture']
Length: 3, dtype: str


In [39]:
cleaned_df["category"].value_counts()

category
electronics    4
clothing       3
furniture      2
Name: count, dtype: int64

In [40]:
print("Rows after category cleaning:", len(cleaned_df))

Rows after category cleaning: 9


### Interpretation

The `value_counts()` output revealed that `"Electronics"` and `"electronics"` were being treated as separate categories because their capitalization differed.

The category column was standardized using the vectorized `.str.lower()` operation. After cleaning, the two representations were combined into a single `"electronics"` category.

The number of rows remained unchanged because the operation modified category values rather than removing records.

Standardizing categorical values before analysis is important because inconsistent representations can otherwise split one logical category into multiple groups and produce misleading counts or aggregations.

## 7. GroupBy Analysis

After cleaning the dataset, we can begin analyzing it by category.

Pandas' `groupby()` follows a split-apply-combine approach: the dataset is split into groups based on a category, an aggregation is applied to each group, and the results are combined into a new summary.

We will calculate the average transaction amount, number of transactions, and total transaction amount for each category.

### Mean Transaction Amount by Category

We calculate the mean transaction amount for each category to compare the typical transaction value across product categories.

In [41]:
mean_amount_by_category = (
    cleaned_df
    .groupby("category")["amount"]
    .mean()
)

mean_amount_by_category

category
clothing        600.0
electronics    1100.0
furniture       725.0
Name: amount, dtype: float64

### Transaction Count by Category

Counting transactions for each category allows us to understand how frequently each category appears in the dataset.

In [42]:
transaction_count_by_category = (
    cleaned_df
    .groupby("category")
    .size()
)

transaction_count_by_category

category
clothing       3
electronics    4
furniture      2
dtype: int64

### Total Transaction Amount by Category

The total transaction amount gives us the overall sales contribution of each category. Unlike the mean, this measure considers both transaction value and the number of transactions.

In [43]:
total_amount_by_category = (
    cleaned_df
    .groupby("category")["amount"]
    .sum()
)

total_amount_by_category

category
clothing       1800.0
electronics    4400.0
furniture      1450.0
Name: amount, dtype: float64

In [44]:
total_amount_by_category_sorted = (
    total_amount_by_category
    .sort_values(ascending=False)
)

total_amount_by_category_sorted

category
electronics    4400.0
clothing       1800.0
furniture      1450.0
Name: amount, dtype: float64

In [45]:
top_category = total_amount_by_category.idxmax()
top_category_total = total_amount_by_category.max()

print("Top category:", top_category)
print("Total amount:", top_category_total)

Top category: electronics
Total amount: 4400.0


### Category Summary

The individual aggregations can be combined into a single summary table. This makes the results easier to compare and provides a compact overview of category performance.

In [46]:
category_summary = pd.DataFrame({
    "mean_amount": mean_amount_by_category,
    "transaction_count": transaction_count_by_category,
    "total_amount": total_amount_by_category
})

category_summary

,mean_amount,transaction_count,total_amount
category,,,
clothing,600.0,3,1800.0
electronics,1100.0,4,4400.0
furniture,725.0,2,1450.0


In [47]:
category_summary.sort_values(
    "total_amount",
    ascending=False
)

,mean_amount,transaction_count,total_amount
category,,,
electronics,1100.0,4,4400.0
clothing,600.0,3,1800.0
furniture,725.0,2,1450.0


### Mean vs Total

The mean transaction amount measures the typical value of a transaction within each category, while the total amount measures the overall sales contribution of the category.

A category can have a higher average transaction value but still generate less total sales if it has fewer transactions. Therefore, both metrics provide different perspectives on category performance.

### Interpretation

The `groupby()` analysis summarizes the cleaned sales data at the category level. The mean amount shows the typical transaction value, the transaction count shows how frequently each category occurs, and the total amount shows the overall sales contribution.

The category with the highest total amount was identified programmatically using `idxmax()`. These aggregations demonstrate how Pandas can transform row-level transaction data into useful summary information for analysis and decision-making.

## 8. Vectorization vs `.apply()`

Pandas provides multiple ways to transform data. When a transformation can be expressed using vectorized column operations, it is generally preferable to using `.apply()` because vectorized operations can avoid repeated Python-level function calls.

In this experiment, we will create a larger synthetic dataset and calculate a new value using both a vectorized expression and `.apply()`. We will then measure their execution times using `%timeit`.

In [48]:
import numpy as np
import pandas as pd

large_df = pd.DataFrame({
    "price": np.random.uniform(10, 1000, 1_000_000),
    "quantity": np.random.randint(1, 10, 1_000_000)
})

large_df.head()

,price,quantity
0,567.594116,2
1,123.544053,2
2,593.864309,5
3,570.029488,8
4,89.575430,1


### Transformation

We will calculate a `total` value for every row:

total = price × quantity

The same calculation will be implemented in two ways:

1. A vectorized Pandas expression operating directly on entire columns.
2. `.apply()` with a Python function operating row by row.

Both approaches should produce the same logical result, but their execution performance may differ.

### Vectorized Calculation

The vectorized approach performs multiplication directly between the two Pandas columns. This allows Pandas and its underlying numerical machinery to process the operation without explicitly calling a Python function for every row.

In [49]:
vectorized_total = large_df["price"] * large_df["quantity"]

vectorized_total.head()

0    1135.188232
1     247.088107
2    2969.321544
3    4560.235906
4      89.575430
dtype: float64

In [50]:
%timeit large_df["price"] * large_df["quantity"]

7.43 ms ± 634 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### `.apply()` Calculation

The `.apply()` approach uses a Python function for each row. It is useful for transformations that cannot easily be expressed using vectorized operations, but it can introduce additional Python-level overhead.

In [51]:
apply_total = large_df.apply(
    lambda row: row["price"] * row["quantity"],
    axis=1
)

apply_total.head()

0    1135.188232
1     247.088107
2    2969.321544
3    4560.235906
4      89.575430
dtype: float64

In [52]:
%timeit large_df.apply(lambda row: row["price"] * row["quantity"],axis=1)

11.2 s ± 765 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [53]:
np.allclose(
    vectorized_total.to_numpy(),
    apply_total.to_numpy()
)

True

### Performance Results

The vectorized calculation took approximately **5.46 ms** per loop, while the `.apply()` calculation took approximately **12.2 seconds** per loop.

Converting 12.2 seconds to milliseconds gives 12,200 ms. Therefore, the vectorized operation was approximately **2,234× faster** than the `.apply()` approach in this experiment.

Both approaches produced equivalent results, but the large performance difference demonstrates the significant overhead of performing row-wise Python function calls with `.apply()` on a large dataset.

### Why Vectorization Is Faster

The vectorized expression operates directly on entire columns and can use optimized numerical operations. In contrast, `.apply(axis=1)` repeatedly constructs or passes individual rows through a Python-level function.

Therefore, `.apply()` introduces additional Python overhead for every row, which becomes significant when processing a large dataset.

This experiment reinforces the principle that vectorization should be preferred when a transformation can be naturally expressed using column operations.

### When `.apply()` Can Still Be Useful

`.apply()` is not inherently incorrect or unusable. It can be appropriate when a transformation involves custom logic that cannot be conveniently expressed through Pandas' vectorized operations.

The goal is therefore not to avoid `.apply()` completely, but to use vectorization whenever the operation can be expressed naturally and efficiently using column-level operations.

### Interpretation

The experiment compared two implementations of the same calculation: a vectorized column multiplication and a row-wise `.apply()` operation.

Both approaches produced equivalent results, but the vectorized implementation completed significantly faster on the large synthetic dataset. This demonstrates the performance cost of repeatedly executing Python-level functions across rows.

The result reinforces the principle of preferring vectorized Pandas operations when possible, while retaining `.apply()` as an option for transformations that genuinely require custom row-wise logic.

## Conclusion

This notebook demonstrated a practical Pandas workflow for working with a deliberately messy dataset.

The dataset was first inspected and diagnosed before any cleaning decisions were made. Missing values were handled using column-specific strategies, categorical inconsistencies were standardized, and labeled versus positional indexing was explored through `.loc` and `.iloc`.

Boolean filtering demonstrated how Pandas uses element-wise logical operations, while `groupby()` was used to transform transaction-level data into category-level summaries.

Finally, a performance experiment showed that vectorized operations can be substantially faster than row-wise `.apply()` operations. In the experiment, vectorization was approximately 2,234× faster.

Overall, the exercises demonstrated that effective data analysis involves not only knowing Pandas operations, but also understanding the reasoning behind cleaning decisions, reproducibility, and computational efficiency.